<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/ltx2.3_all_guanfang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1: 安装 ComfyUI + 满足全部上传 LTX-2.3 工作流的精简节点
# ==========================================
import os
import subprocess

print("=== 🚀 开始安装 ComfyUI 基础环境与精简节点 ===")

%cd /content

# --------------------------
# 1. 安装 / 更新 ComfyUI
# --------------------------
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    !cd ComfyUI && git pull

%cd /content/ComfyUI

# ComfyUI 基础依赖
!pip install -q -r requirements.txt
!pip install -q huggingface_hub hf_transfer

comfyui_dir = "/content/ComfyUI"
custom_nodes_dir = os.path.join(comfyui_dir, "custom_nodes")
os.makedirs(custom_nodes_dir, exist_ok=True)

# --------------------------
# 2. 精简节点清单
# --------------------------
node_repos = [
    # 管理器：建议保留，便于检查缺失节点
    "https://github.com/ltdrdata/ComfyUI-Manager.git",

    # LTX 2.3 全部核心节点：
    # LTXV、IC-LoRA、Audio VAE、Gemma Text Encoder、HDR、LipDub、Motion Track、Inpaint、Outpaint 等
    "https://github.com/Lightricks/ComfyUI-LTXVideo.git",

    # 视频读写：
    # LoadVideo / GetVideoComponents / CreateVideo / SaveVideo
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",

    # 常用图像、Mask、Resize 辅助节点：
    # ResizeImageMaskNode / GetImageSize 等
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/cubiq/ComfyUI_essentials.git",

    # Union Control 工作流需要：
    # DWPreprocessor
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",

    # HDR 工作流需要：
    # SimpleMath+
    "https://github.com/evanspearman/ComfyMath.git",

    # 前端辅助节点，建议保留
    "https://github.com/rgthree/rgthree-comfy.git",

    # ==================================================
    # 以下当前上传的工作流不需要，已注释
    # ==================================================
    # "https://github.com/yuvraj108c/ComfyUI-Video-Depth-Anything.git",
    # "https://github.com/chrisgoringe/cg-use-everywhere.git",
    # "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    # "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    # "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    # "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    # "https://github.com/DoctorDiffusion/ComfyUI-MediaMixer.git",
    # "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git",
    # "https://github.com/11cafe/comfyui-workspace-manager.git",
    # "https://github.com/kijai/ComfyUI-PromptRelay.git",
    # "https://github.com/Saganaki22/ComfyUI-FishAudioS2.git",
    # "https://github.com/WhatDreamsCost/WhatDreamsCost-ComfyUI.git",
]

def run_cmd(cmd, cwd=None, quiet=False):
    result = subprocess.run(
        cmd,
        cwd=cwd,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print("❌ 命令失败:", " ".join(cmd))
        if result.stderr:
            print(result.stderr.strip())
    else:
        if not quiet and result.stdout.strip():
            print(result.stdout.strip())
    return result

def install_node(repo_url):
    repo_name = repo_url.split("/")[-1].replace(".git", "")
    repo_path = os.path.join(custom_nodes_dir, repo_name)

    if not os.path.exists(repo_path):
        print(f"\n📥 正在克隆: {repo_name}")
        result = run_cmd(["git", "clone", repo_url, repo_path])
        if result.returncode != 0:
            print(f"⚠️ 跳过依赖安装: {repo_name}")
            return
        print(f"✅ 克隆成功: {repo_name}")
    else:
        print(f"\n✅ 已存在: {repo_name}，正在更新")
        result = run_cmd(["git", "pull"], cwd=repo_path)
        if result.returncode == 0:
            print(f"🔄 更新完成: {repo_name}")

    req_file = os.path.join(repo_path, "requirements.txt")
    if os.path.exists(req_file):
        print(f"⚙️ 安装依赖: {repo_name}")
        subprocess.run(["pip", "install", "-q", "-r", req_file])
    else:
        print(f"⏭️ 无 requirements.txt: {repo_name}")

print("\n=== 🚀 开始安装精简节点 ===")
for repo in node_repos:
    install_node(repo)

# --------------------------
# 3. LTXVideo 依赖修复
# --------------------------
print("\n=== 🔧 修复 LTXVideo 依赖 ===")
subprocess.run(["pip", "install", "-q", "kornia==0.7.2"])

ltx_req = os.path.join(custom_nodes_dir, "ComfyUI-LTXVideo", "requirements.txt")
if os.path.exists(ltx_req):
    subprocess.run(["pip", "install", "-q", "-r", ltx_req])

# --------------------------
# 4. 全局基础依赖
# --------------------------
print("\n=== 📦 安装全局基础依赖 ===")
subprocess.run([
    "pip", "install", "-q",
    "opencv-python-headless",
    "scikit-image",
    "onnxruntime-gpu",
    "sageattention"
])

print("\n🎉 Cell 1 完成：ComfyUI 与精简节点安装完成！")

In [ ]:
# ==========================================
# Cell 2: 下载满足全部上传 LTX-2.3 工作流的必要模型
# ==========================================
import os
import shutil
from google.colab import userdata
from huggingface_hub import hf_hub_download, snapshot_download
from concurrent.futures import ThreadPoolExecutor

print("=== 🚀 开始下载 LTX-2.3 工作流必要模型 ===")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("✅ 已读取 HF_TOKEN")
    else:
        raise Exception("HF_TOKEN 为空")
except Exception as e:
    raise Exception("❌ 请在 Colab 左侧 Secrets 中添加 HF_TOKEN") from e

comfyui_dir = "/content/ComfyUI"

# --------------------------
# 1. 精确文件下载任务
# --------------------------
file_downloads = [
    # 主模型：所有工作流共用
    {
        "repo_id": "Lightricks/LTX-2.3",
        "filename": "ltx-2.3-22b-dev.safetensors",
        "local_dir": f"{comfyui_dir}/models/checkpoints"
    },

    # Distilled LoRA：全部 distilled 工作流共用
    {
        "repo_id": "Lightricks/LTX-2.3",
        "filename": "ltx-2.3-22b-distilled-lora-384-1.1.safetensors",
        "local_dir": f"{comfyui_dir}/models/loras/ltxv/ltx2"
    },

    # Pixel Spatial Upscaler 工作流需要
    {
        "repo_id": "Lightricks/LTX-2.3",
        "filename": "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
        "local_dir": f"{comfyui_dir}/models/latent_upscale_models"
    },

    # Gemma 文本编码器
    {
        "repo_id": "Comfy-Org/ltx-2",
        "filename": "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors",
        "local_dir": f"{comfyui_dir}/models/text_encoders"
    },

    # LTX 2.3 text projection
    {
        "repo_id": "Kijai/LTX2.3_comfy",
        "filename": "text_encoders/ltx-2.3_text_projection_bf16.safetensors",
        "local_dir": f"{comfyui_dir}/models/text_encoders"
    },

    # ==================================================
    # 以下当前上传的工作流未明确需要，已注释
    # ==================================================
    # {
    #     "repo_id": "Lightricks/LTX-2.3-fp8",
    #     "filename": "ltx-2.3-22b-dev-fp8.safetensors",
    #     "local_dir": f"{comfyui_dir}/models/checkpoints"
    # },
    # {
    #     "repo_id": "Comfy-Org/ltx-2.3",
    #     "filename": "split_files/loras/ltx-2.3-id-lora-talkvid-3k.safetensors",
    #     "local_dir": f"{comfyui_dir}/models/loras"
    # },
    # {
    #     "repo_id": "stronman/LTX-2.3-Transition-LORA",
    #     "filename": "ltx2.3-transition.safetensors",
    #     "local_dir": f"{comfyui_dir}/models/loras"
    # },
    # {
    #     "repo_id": "stronman/Ltx2.3-VBVR-lora-I2V",
    #     "filename": "Ltx2.3-Licon-VBVR-I2V-390K-R32.safetensors",
    #     "local_dir": f"{comfyui_dir}/models/loras"
    # },
]

# --------------------------
# 2. IC-LoRA 仓库下载任务
# 只下载 safetensors，放进工作流常用路径 models/loras/ltxv/ltx2
# --------------------------
ic_lora_repos = [
    # Inpaint / Outpaint
    "Lightricks/LTX-2.3-22b-IC-LoRA-In-Outpainting",

    # Union Control
    "Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control",

    # HDR
    "Lightricks/LTX-2.3-22b-IC-LoRA-HDR",

    # Motion Track
    "Lightricks/LTX-2.3-22b-IC-LoRA-Motion-Track-Control",

    # LipDub
    "Lightricks/LTX-2.3-22b-IC-LoRA-LipDub",

    # Ingredients / Reference Sheet Control
    "Lightricks/LTX-2.3-22b-IC-LoRA-Ingredients",
]

loras_ltx_dir = f"{comfyui_dir}/models/loras/ltxv/ltx2"
os.makedirs(loras_ltx_dir, exist_ok=True)

def download_file_task(task):
    try:
        os.makedirs(task["local_dir"], exist_ok=True)
        hf_hub_download(
            repo_id=task["repo_id"],
            filename=task["filename"],
            local_dir=task["local_dir"],
            repo_type="model"
        )
        print(f"✅ 文件完成: {task['filename']}")
    except Exception as e:
        print(f"❌ 文件失败: {task['repo_id']} / {task['filename']}")
        print(f"   错误: {e}")

def download_ic_lora_repo(repo_id):
    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=loras_ltx_dir,
            local_dir_use_symlinks=False,
            allow_patterns=["*.safetensors", "*.json", "*.txt", "*.md"]
        )
        print(f"✅ IC-LoRA 完成: {repo_id}")
    except Exception as e:
        print(f"❌ IC-LoRA 失败: {repo_id}")
        print(f"   错误: {e}")

print(f"\n🚀 开始下载 {len(file_downloads)} 个精确模型文件...")
with ThreadPoolExecutor(max_workers=5) as executor:
    list(executor.map(download_file_task, file_downloads))

print(f"\n🚀 开始下载 {len(ic_lora_repos)} 个 IC-LoRA 仓库的 safetensors...")
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(download_ic_lora_repo, ic_lora_repos))

print("\n🎉 Cell 2 完成：模型下载流程结束！")

In [ ]:
# ==========================================
# Cell 3: 模型文件名与路径兼容修复
# ==========================================
import os
import shutil
import glob

print("=== 🔧 开始修复模型文件名与路径兼容 ===")

comfyui_dir = "/content/ComfyUI"

text_encoder_dir = f"{comfyui_dir}/models/text_encoders"
loras_root = f"{comfyui_dir}/models/loras"
loras_ltx_dir = f"{comfyui_dir}/models/loras/ltxv/ltx2"

os.makedirs(text_encoder_dir, exist_ok=True)
os.makedirs(loras_root, exist_ok=True)
os.makedirs(loras_ltx_dir, exist_ok=True)

# --------------------------
# 1. Gemma 文本编码器别名
# 工作流常见名称：comfy_gemma_3_12B_it.safetensors
# --------------------------
gemma_candidates = [
    f"{text_encoder_dir}/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors",
    f"{text_encoder_dir}/gemma_3_12B_it_fp4_mixed.safetensors",
]

gemma_dst = f"{text_encoder_dir}/comfy_gemma_3_12B_it.safetensors"

for src in gemma_candidates:
    if os.path.exists(src):
        if not os.path.exists(gemma_dst):
            shutil.copy(src, gemma_dst)
            print("✅ 已创建 Gemma 别名: comfy_gemma_3_12B_it.safetensors")
        else:
            print("✅ Gemma 别名已存在")
        break
else:
    print("⚠️ 未找到 Gemma 源文件，请检查 Cell 2 下载日志")

# --------------------------
# 2. Text projection 如果在子目录，复制到 text_encoders 根目录
# --------------------------
projection_candidates = [
    f"{text_encoder_dir}/text_encoders/ltx-2.3_text_projection_bf16.safetensors",
    f"{text_encoder_dir}/ltx-2.3_text_projection_bf16.safetensors",
]

projection_dst = f"{text_encoder_dir}/ltx-2.3_text_projection_bf16.safetensors"

for src in projection_candidates:
    if os.path.exists(src):
        if src != projection_dst and not os.path.exists(projection_dst):
            shutil.copy(src, projection_dst)
            print("✅ 已复制 text projection 到 text_encoders 根目录")
        else:
            print("✅ text projection 已存在")
        break

# --------------------------
# 3. LoRA 路径双向兼容
# 有些工作流读取 ltxv/ltx2/xxx，有些菜单可能显示根目录
# 这里把 ltxv/ltx2 下的 safetensors 同步一份到 loras 根目录
# 只复制不存在的文件
# --------------------------
lora_files = glob.glob(f"{loras_ltx_dir}/**/*.safetensors", recursive=True)

for src in lora_files:
    filename = os.path.basename(src)
    dst = os.path.join(loras_root, filename)
    if not os.path.exists(dst):
        try:
            shutil.copy(src, dst)
            print(f"✅ LoRA 根目录兼容: {filename}")
        except Exception as e:
            print(f"⚠️ LoRA 复制失败: {filename} -> {e}")

# --------------------------
# 4. 反向兼容：如果有 LoRA 在根目录，也复制到 ltxv/ltx2
# --------------------------
root_loras = glob.glob(f"{loras_root}/*.safetensors")

for src in root_loras:
    filename = os.path.basename(src)
    dst = os.path.join(loras_ltx_dir, filename)
    if not os.path.exists(dst):
        try:
            shutil.copy(src, dst)
            print(f"✅ LoRA ltxv/ltx2 兼容: {filename}")
        except Exception as e:
            print(f"⚠️ LoRA 复制失败: {filename} -> {e}")

# --------------------------
# 5. 打印关键模型检查结果
# --------------------------
required_paths = [
    f"{comfyui_dir}/models/checkpoints/ltx-2.3-22b-dev.safetensors",
    f"{loras_ltx_dir}/ltx-2.3-22b-distilled-lora-384-1.1.safetensors",
    f"{comfyui_dir}/models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
    f"{text_encoder_dir}/comfy_gemma_3_12B_it.safetensors",
    f"{text_encoder_dir}/ltx-2.3_text_projection_bf16.safetensors",
]

print("\n=== 🔍 关键模型检查 ===")
for path in required_paths:
    if os.path.exists(path):
        print(f"✅ 存在: {path}")
    else:
        print(f"❌ 缺失: {path}")

print("\n=== 🔍 已安装 IC-LoRA safetensors ===")
for path in sorted(glob.glob(f"{loras_ltx_dir}/*.safetensors")):
    print(" -", os.path.basename(path))

print("\n🎉 Cell 3 完成：路径兼容修复结束！")

In [ ]:
# ==========================================
# Cell 4: FRP 内网穿透配置
# ==========================================
import os
from google.colab import userdata

print("=== 🚀 开始配置 FRP ===")

try:
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")

    if not VPS_IP or not FRP_TOKEN:
        raise Exception("VPS_IP 或 FRP_TOKEN 为空")
except:
    raise Exception("❌ 请在 Colab 左侧 Secrets 中添加 VPS_IP 和 FRP_TOKEN")

%cd /content

# 下载 FRP
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    !wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content

frpc_conf = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
"""

conf_path = "/content/frp_0.56.0_linux_amd64/frpc.toml"

with open(conf_path, "w") as f:
    f.write(frpc_conf.strip())

print("✅ FRP 配置完成！")
print(f"📄 配置文件: {conf_path}")

In [ ]:
# ==========================================
# Cell 5: 启动 FRP + ComfyUI
# ==========================================
import os
import subprocess
import threading
import time
import configparser

print("=== 🚀 启动 FRP 与 ComfyUI ===")

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

# --------------------------
# 1. Colab 保活
# --------------------------
def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] Colab 保持连接活跃中...")

print("⏳ 启动 Colab 保活线程...")
threading.Thread(target=keep_alive, daemon=True).start()

# --------------------------
# 2. 启动 FRP
# --------------------------
def start_frpc():
    frpc_bin = "/content/frp_0.56.0_linux_amd64/frpc"
    frpc_conf = "/content/frp_0.56.0_linux_amd64/frpc.toml"

    if not os.path.exists(frpc_bin):
        print("❌ 未找到 frpc，请先运行 Cell 4")
        return

    subprocess.run([frpc_bin, "-c", frpc_conf])

print("⏳ 正在后台启动 FRP...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透线程已启动")
print("👉 ComfyUI 启动完成后访问: http://cjp.usdream.dpdns.org:8090")
print("============================================================\n")

# --------------------------
# 3. 配置 ComfyUI-Manager 为 private，跳过启动时 Fetch
# --------------------------
print("⏳ 配置 ComfyUI-Manager 网络模式为 private...")

manager_config_paths = [
    "/content/ComfyUI/user/__manager/config.ini",
    "/content/ComfyUI/user/default/ComfyUI-Manager/config.ini",
    "/content/ComfyUI/custom_nodes/ComfyUI-Manager/config.ini"
]

for config_path in manager_config_paths:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)

    config = configparser.ConfigParser()

    if os.path.exists(config_path):
        config.read(config_path)

    if "default" not in config:
        config["default"] = {}

    config["default"]["network_mode"] = "private"

    with open(config_path, "w") as f:
        config.write(f)

print("✅ Manager private 模式配置完成\n")

# --------------------------
# 4. 启动 ComfyUI
# --------------------------
print("⏳ 正在启动 ComfyUI 主进程...\n")

%cd /content/ComfyUI

# 如果显存较紧，可以加参数：
# --lowvram
# --preview-method auto
!python main.py --dont-print-server